The goal of this code is to scrape the whole text of the documents in "complete_data_set7May2024.csv". I want to get the full text and the antecentes section when is it available. And to do so, I need to first get the link to https://hj.tribunalconstitucional.es/ because there's no way to reconstruct the link using the data in "complete_data_set7May2024.csv".

In [1]:
import re
import json
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')
from tqdm.notebook import tqdm

In [2]:
# Load Abstract Review
ar_data = pd.read_csv("DB_AbstractReviewSpainRodilla_R.csv", sep=None, engine='python', on_bad_lines='skip')
# Load Concrete Review
cr_data = pd.read_csv("DB_RecursosAmparo80_24.csv", sep=None, engine='python', on_bad_lines='skip')
# Load GitHub Data
gh_data = pd.read_csv("complete_data_set7May2024.csv", sep=None, engine='python', on_bad_lines='skip')

print(f"AR rows: {len(ar_data)}, CR rows: {len(cr_data)}, GH rows: {len(gh_data)}")

AR rows: 1734, CR rows: 21782, GH rows: 117689


In [3]:
# Clean the identifyers to get only AUTO XX/XXXX or SENTENCIA XX/XXXX
# The regex handles:
#   - writing/encoding variants (case-insensitive)
#   - any amount of surrounding whitespace or boilerplate text
#   - variable-length numbers on both sides of the slash

def extract_id(text):
    if pd.isna(text):
        return None
    m = re.search(r'((?:SENTENCIA|AUTO)\s+\d+/\d+)', str(text), re.IGNORECASE)
    if m:
        # Normalise to uppercase with a single space
        return re.sub(r'\s+', ' ', m.group(1).upper().strip())
    return None

# Clean Abstract Review: 'id_resolution'
ar_data['clean_id'] = ar_data['id_resolution'].apply(extract_id)

# Clean Concrete Review: 'dicta_names'
cr_data['clean_id'] = cr_data['dicta_names'].apply(extract_id)

print(f"AR: {ar_data['clean_id'].notna().sum()} / {len(ar_data)} IDs extracted")
print(f"CR: {cr_data['clean_id'].notna().sum()} / {len(cr_data)} IDs extracted")
print("\nAR examples:", ar_data['clean_id'].dropna().head(3).tolist())
print("CR examples:", cr_data['clean_id'].dropna().head(3).tolist())

AR: 1734 / 1734 IDs extracted
CR: 21782 / 21782 IDs extracted

AR examples: ['SENTENCIA 191/2016', 'SENTENCIA 190/2016', 'SENTENCIA 186/2016']
CR examples: ['AUTO 116/1980', 'AUTO 115/1980', 'AUTO 114/1980']


In [4]:
# Append links to the GitHub database (copy) from AR and CR using clean_id / ID_PAT

BASE = "https://hj.tribunalconstitucional.es"

# AR already stores full URLs in 'link_id'
ar_lookup = (
    ar_data.dropna(subset=['clean_id', 'link_id'])
    .drop_duplicates('clean_id')
    .set_index('clean_id')['link_id']
    .to_dict()
)

# CR stores relative paths in 'dicta'; prepend the base URL
cr_data['full_link'] = BASE + cr_data['dicta'].astype(str)
cr_lookup = (
    cr_data.dropna(subset=['clean_id'])
    .drop_duplicates('clean_id')
    .set_index('clean_id')['full_link']
    .to_dict()
)

# Merge; AR takes priority since it already holds verified full URLs
link_lookup = {**cr_lookup, **ar_lookup}

gh_copy = gh_data.copy()
gh_copy['link'] = gh_copy['ID_PAT'].map(link_lookup)

# Check how many IDs were not found — group by ID_PAT since each document
# has multiple rows (one per judge-vote) and the meaningful unit is the document
id_link = gh_copy.groupby('ID_PAT')['link'].first()
n_total   = len(id_link)
n_missing = id_link.isna().sum()
n_found   = n_total - n_missing

print(f"Unique documents total:     {n_total:>7}")
print(f"Unique documents with link: {n_found:>7}  ({n_found/n_total*100:.1f}%)")
print(f"Unique documents missing:   {n_missing:>7}  ({n_missing/n_total*100:.1f}%)")
print("\nSample unmatched IDs:")
print(id_link[id_link.isna()].index[:10].tolist())

Unique documents total:       23844
Unique documents with link:   23452  (98.4%)
Unique documents missing:       392  (1.6%)

Sample unmatched IDs:
['AUTO 1/2002', 'AUTO 10/2002', 'AUTO 101/2002', 'AUTO 102/2002', 'AUTO 103/1981 bis', 'AUTO 103/2002', 'AUTO 104/2002', 'AUTO 104/2004 bis', 'AUTO 105/2002', 'AUTO 106/2002']


In [5]:
# Use the links to retrieve, through the API, the 'antecedentes' section
# when available, and the full text for the rest.
# Safe to re-run: already-cached JSONs are read from disk, already-processed
# IDs are skipped, and results are appended to the CSV incrementally.

API     = "https://hj.tribunalconstitucional.es/Resolucion/Api/json/{id}"
CACHE   = Path("tc_results")
CACHE.mkdir(exist_ok=True)
OUT_CSV = Path("retrieved_texts.csv")

def numeric_id_from_link(url):
    m = re.search(r'/(\d+)$', str(url))
    return int(m.group(1)) if m else None

def get_json(numeric_id):
    fpath = CACHE / f"{numeric_id}.json"
    if fpath.exists():
        return json.loads(fpath.read_text(encoding='utf-8'))
    r = requests.get(API.format(id=numeric_id), timeout=15)
    data = json.loads(r.content.decode('utf-8'))
    fpath.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
    time.sleep(0.3)
    return data

def section_text(section_list):
    if not section_list:
        return ""
    return "\n\n".join(item.get('TEXTO', '') for item in section_list if item.get('TEXTO'))

def extract_texts(data):
    antecedentes = section_text(data.get('RESOLUCIONES_ANTECEDENTES', []))
    full_text = "\n\n".join(filter(None, [
        section_text(data.get('RESOLUCIONES_CABECERA', [])),
        section_text(data.get('RESOLUCIONES_ANTECEDENTES', [])),
        section_text(data.get('RESOLUCIONES_FUNDAMENTOS', [])),
        section_text(data.get('RESOLUCIONES_DICTAMEN', [])),
        section_text(data.get('RESOLUCIONES_PIE', [])),
    ]))
    return antecedentes or None, full_text

# All unique documents that have a link
unique_docs = (
    gh_copy.dropna(subset=['link'])[['ID_PAT', 'link']]
    .drop_duplicates('ID_PAT')
    .reset_index(drop=True)
)

# Skip IDs already present in the output CSV (resumability)
if OUT_CSV.exists():
    already_done = set(pd.read_csv(OUT_CSV, usecols=['ID_PAT'])['ID_PAT'])
    todo = unique_docs[~unique_docs['ID_PAT'].isin(already_done)].reset_index(drop=True)
    print(f"Already processed: {len(already_done)} — remaining: {len(todo)}")
else:
    todo = unique_docs
    print(f"Starting fresh — documents to retrieve: {len(todo)}")

errors = []

for _, row in tqdm(todo.iterrows(), total=len(todo), desc="Retrieving"):
    nid = numeric_id_from_link(row['link'])
    if nid is None:
        errors.append({'ID_PAT': row['ID_PAT'], 'error': 'could not parse numeric id'})
        continue
    try:
        data = get_json(nid)
        antecedentes, full_text = extract_texts(data)
        result = pd.DataFrame([{
            'ID_PAT':       row['ID_PAT'],
            'numeric_id':   nid,
            'antecedentes': antecedentes,
            'full_text':    full_text,
        }])
        # Append one row at a time so progress is never lost on interruption
        result.to_csv(OUT_CSV, mode='a', header=not OUT_CSV.exists(), index=False)
    except Exception as e:
        errors.append({'ID_PAT': row['ID_PAT'], 'numeric_id': nid, 'error': str(e)})

# Summary
texts_df = pd.read_csv(OUT_CSV)
print(f"\nTotal in CSV:       {len(texts_df)}")
print(f"With antecedentes:  {texts_df['antecedentes'].notna().sum()}")
print(f"Errors this run:    {len(errors)}")
if errors:
    print("\nFirst 5 errors:")
    for e in errors[:5]:
        print(" ", e)

Already processed: 7188 — remaining: 16264


Retrieving:   0%|          | 0/16264 [00:00<?, ?it/s]


Total in CSV:       23445
With antecedentes:  18948
Errors this run:    7

First 5 errors:
  {'ID_PAT': 'AUTO 274/1987', 'numeric_id': 11408, 'error': "HTTPSConnectionPool(host='hj.tribunalconstitucional.es', port=443): Max retries exceeded with url: /Resolucion/Api/json/11408 (Caused by ConnectTimeoutError(<HTTPSConnection(host='hj.tribunalconstitucional.es', port=443) at 0x1b06e33d6d0>, 'Connection to hj.tribunalconstitucional.es timed out. (connect timeout=15)'))"}
  {'ID_PAT': 'AUTO 273/1987', 'numeric_id': 11407, 'error': "HTTPSConnectionPool(host='hj.tribunalconstitucional.es', port=443): Max retries exceeded with url: /Resolucion/Api/json/11407 (Caused by ConnectTimeoutError(<HTTPSConnection(host='hj.tribunalconstitucional.es', port=443) at 0x1b06e33da90>, 'Connection to hj.tribunalconstitucional.es timed out. (connect timeout=15)'))"}
  {'ID_PAT': 'AUTO 965/1988', 'numeric_id': 13540, 'error': "HTTPSConnectionPool(host='hj.tribunalconstitucional.es', port=443): Max retries exc

In [6]:
texts_df.to_csv("Full_text_data_11May26.csv")

In [7]:
texts_df["full_text"].isna().sum()

np.int64(4497)

In [ ]:
# Missing:
# 4497 (19%)
# Distinct:
# 18947 (81%)

hola
